In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
import os, json

In [2]:
TONE_DIR = os.path.join('skin_dataset', 'Skintone')

# automatically get all class folders
classes = [d for d in os.listdir(TONE_DIR) if os.path.isdir(os.path.join(TONE_DIR, d))]

for cls in classes:
    path = os.path.join(TONE_DIR, cls)
    num_images = len([f for f in os.listdir(path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
    print(f"{cls}: {num_images} images")


dark: 30 images
light: 30 images
mid-dark: 30 images
mid-light: 30 images


In [3]:
# %% settings
TONE_DIR = os.path.join('skin_dataset','Skintone')
IMAGE_SIZE = (224,224)
BATCH_SIZE = 32
EPOCHS = 30
MODEL_OUT = 'model_skintone.h5'


# %% data
train_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2,
rotation_range=15, horizontal_flip=True)
train_gen = train_datagen.flow_from_directory(
TONE_DIR,
target_size=IMAGE_SIZE,
batch_size=BATCH_SIZE,
class_mode='categorical',
subset='training'
)
val_gen = train_datagen.flow_from_directory(
TONE_DIR,
target_size=IMAGE_SIZE,
batch_size=BATCH_SIZE,
class_mode='categorical',
subset='validation'
)


# %% model
base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(*IMAGE_SIZE,3))
base.trainable = False
x = layers.GlobalAveragePooling2D()(base.output)
x = layers.Dense(128, activation='relu')(x)
output = layers.Dense(train_gen.num_classes, activation='softmax')(x)
model = models.Model(inputs=base.input, outputs=output)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


# %% train
history = model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS)


# %% save
model.save(MODEL_OUT)
with open('label_map_skintone.json','w') as f:
    json.dump(train_gen.class_indices, f)
print('Saved', MODEL_OUT)

Found 96 images belonging to 4 classes.
Found 24 images belonging to 4 classes.


C:\Users\A S U S\anaconda3\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 12s 2s/step - accuracy: 0.5130 - loss: 1.2186 - val_accuracy: 0.6250 - val_loss: 1.1615
Epoch 2/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 1s/step - accuracy: 0.5729 - loss: 0.9935 - val_accuracy: 0.4583 - val_loss: 1.1708
Epoch 3/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 2s/step - accuracy: 0.7070 - loss: 0.7338 - val_accuracy: 0.4583 - val_loss: 1.0799
Epoch 4/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 1s/step - accuracy: 0.7695 - loss: 0.5890 - val_accuracy: 0.5000 - val_loss: 1.1809
Epoch 5/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 1s/step - accuracy: 0.7917 - loss: 0.5080 - val_accuracy: 0.5000 - val_loss: 1.2516
Epoch 6/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 1s/step - accuracy: 0.8320 - loss: 0.4506 - val_accuracy: 0.6667 - val_loss: 1.0242
Epoch 7/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 1s/step - accuracy: 0.8594 - loss: 0.4062 - val_accuracy: 0.5417 - val_loss: 1.1635
Epoch 8/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 1s/step - accuracy: 0.8750 - loss: 0.3585 - val_accuracy: 0.5417 - val_loss: 1.1084
Epoch 9/30
3/3 

Saved model_skintone.h5
